# The hold-out track record


# The hold-out track record

Every labelled gene was hidden once and each strategy that calls labels was asked what it is. This notebook reads the record that ships with Starplast and states what it says. It was built by `scripts/build_track_record.py` (`starplast/track_record.py`), design in `instructions/open/62_holdout_track_record.md`.

Three rules make it trustworthy rather than merely present: the five folds never split an orthogroup (a paralogue cannot give away its sibling's label); a strategy that says nothing is counted apart from one that is wrong; and no percentage is printed from fewer than five answered genes.

In [1]:
import pandas as pd
from starplast import track_record as T
led = pd.read_parquet(T.shipped_path())
led.groupby(['organism', 'mode'], observed=True).size().rename('rows')

organism  mode    
Pf        set         231414
          together    136262
Tg        set         415588
          together    255108

## Every recorded label

Each organism's biological labels (`track_record.labels`): the declared targets, and for Toxoplasma the cell-cycle phase and four specific screen phenotypes. Provenance columns are left out on purpose. Per label: the best strategy on mean per-class recall (small classes count equally), its overall accuracy, and what always naming the commonest class scores.

In [2]:
rows = []
for (org, target), part in led[led['mode'] == 'together'].groupby(['organism', 'target'], observed=True):
    every = part.drop_duplicates('gene')['truth'].astype(str)
    best = max(((T._baselines(p), k) for k, p in part.groupby(part['strategy'].astype(str))
                if T._baselines(p) is not None), key=lambda t: t[0]['balanced'])
    rows.append({'organism': org, 'label': target, 'genes': len(every), 'classes': every.nunique(),
                 'best strategy': best[1], 'per class': best[0]['balanced'],
                 'chance': 1 / every.nunique(), 'overall': best[0]['accuracy'],
                 'commonest': every.value_counts(normalize=True).iloc[0]})
pd.DataFrame(rows).round(2)

,organism,label,genes,classes,best strategy,per class,chance,overall,commonest
0,Pf,is_exported,5720,2,stacking,0.85,0.5,0.92,0.97
1,Pf,lopit_pf_location,1580,18,triangulation,0.64,0.06,0.69,0.24
2,Pf,pb_transferred_phenotype,2433,3,stacking,0.56,0.33,0.61,0.48
3,Tg,cellcycle_phase,873,5,stacking,0.6,0.2,0.6,0.3
4,Tg,compartment,3804,24,physical_partners,0.54,0.04,0.56,0.2
5,Tg,dtm_class,8137,4,stacking,0.64,0.25,0.72,0.74
6,Tg,lopit_unified,3818,10,physical_partners,0.71,0.1,0.7,0.33
7,Tg,screen_actin_phenotype,318,2,layer_propagation,0.57,0.5,0.78,0.86
8,Tg,screen_any_phenotype,318,2,structural_homology,0.66,0.5,0.69,0.69
9,Tg,screen_apicoplast_phenotype,318,2,structural_homology,0.68,0.5,0.76,0.81


## Coverage

`together` rows are the fold hold-outs: one per (strategy, labelled gene). `set` rows hide several genes at once. Each fold-held gene must appear once per strategy -- no more, no less.

In [3]:
folds = led[led['mode'] == 'together']
per = folds.groupby(['organism', 'target', 'strategy'], observed=True)['gene'].agg(['size', 'nunique'])
assert (per['size'] == per['nunique']).all(), 'a gene was held out twice'
per.rename(columns={'size': 'held out', 'nunique': 'distinct genes'})

held out  distinct genes
organism target            strategy                                       
Pf       is_exported       cluster_guilt              5720            5720
                           feature_knn                5720            5720
                           graph_convolution          5720            5720
                           layer_propagation          5720            5720
                           layer_vote                 5720            5720
                           map_neighbours             5720            5720
                           physical_partners          5720            5720
                           random_forest              5720            5720
                           stacking                   5720            5720
                           stratum_focus              5720            5720
                           structural_homology        5720            5720
                           supervised_classifier      5720            5720
                           triangulation              5720            5720
                           understudied_first         5720            5720
         lopit_pf_location cluster_guilt              1580            1580
                           feature_knn                1580            1580
                           graph_convolution          1580            1580
                           layer_propagation          1580            1580
                           layer_vote                 1580            1580
                           map_neighbours             1580            1580

The strategy and class levels below are each organism's main label (`T.default_target`); the table above covers the rest.

In [4]:
main = pd.concat([folds[(folds['organism'] == o) & (folds['target'].astype(str) == T.default_target(o, led))]
                  for o in folds['organism'].astype(str).unique()])
main.groupby('organism', observed=True)['target'].first()

organism
Pf    lopit_pf_location
Tg          compartment
Categories (12, str): ['cellcycle_phase', 'compartment', 'dtm_class', 'is_exported', ...,
                       'screen_any_phenotype', 'screen_apicoplast_phenotype', 'screen_egress_phenotype',
                       'screen_replication_phenotype']

## Strategy level

Right over answered, with the Wilson 95% interval. `answered` below the number held out means the strategy abstained on the rest -- shown, not hidden in the rate.

In [5]:
s = T.summary(main, 'strategy')
s[['organism', 'strategy', 'answered', 'right', 'wrong', 'abstained', 'rate', 'rate_low', 'rate_high']].sort_values(['organism', 'rate'], ascending=[True, False]).round(3)

,organism,strategy,answered,right,wrong,abstained,rate,rate_low,rate_high
6,Pf,physical_partners,58,48,10,1522,0.828,0.711,0.904
12,Pf,triangulation,1205,831,374,375,0.69,0.663,0.715
13,Pf,understudied_first,1205,831,374,375,0.69,0.663,0.715
7,Pf,random_forest,1580,1020,560,0,0.646,0.622,0.669
1,Pf,feature_knn,1264,777,487,316,0.615,0.588,0.641
9,Pf,stratum_focus,1264,777,487,316,0.615,0.588,0.641
2,Pf,graph_convolution,1580,938,642,0,0.594,0.569,0.618
8,Pf,stacking,1580,932,648,0,0.59,0.565,0.614
11,Pf,supervised_classifier,1580,906,674,0,0.573,0.549,0.598
4,Pf,layer_vote,1580,868,712,0,0.549,0.525,0.574


## Class level: which distinctions the data carries

For each class, the best strategy's rate. A class every strategy misses is a statement about the measurements, not about any one strategy.

In [6]:
c = T.summary(main[main.organism == 'Tg'], 'class')
c = c[c['enough'].astype(bool)]
best = c.sort_values('rate', ascending=False).drop_duplicates('truth')
best[['truth', 'strategy', 'right', 'answered', 'rate', 'confused_with']].round(3)

,truth,strategy,right,answered,rate,confused_with
144,19S proteasome,physical_partners,16,16,1,
148,ER 2,physical_partners,8,8,1,
146,60S ribosome,physical_partners,37,40,0.925,"nucleus - chromatin, nucleus - non-chromatin, cytosol"
150,IMC,physical_partners,34,37,0.919,"cytosol, PM - peripheral 2"
248,PM - peripheral 1,structural_homology,21,23,0.913,"IMC, nucleus - chromatin"
161,mitochondrion - membranes,physical_partners,54,61,0.885,"mitochondrion - soluble, nucleus - chromatin"
160,micronemes,physical_partners,21,24,0.875,"cytosol, ER 2"
158,dense granules,physical_partners,45,56,0.804,"cytosol, ER 2, mitochondrion - soluble"
46,rhoptries 1,feature_knn,36,46,0.783,"rhoptries 2, nucleus - chromatin, dense granules"
44,nucleus - chromatin,feature_knn,463,634,0.73,"nucleus - non-chromatin, cytosol, mitochondrion - soluble"


## Hidden together: does it still work when many genes are unknown?

Random sets of 1 to 500 genes hidden at once. A strategy that only works when a gene's neighbours are labelled falls as the set grows; a flat curve means the evidence reaches the genes themselves.

In [7]:
sets = led[(led['mode'] == 'set') & led['set_name'].astype(str).str.startswith('random')]
curve = sets.assign(answered=~sets['abstained'].astype(bool))
curve = curve[curve['answered']].groupby(['organism', 'target', 'strategy', 'set_size'], observed=True)['correct'].mean().unstack('set_size')
curve.round(2)

set_size                                           1         5         20        100       500
organism target            strategy                                                           
Pf       is_exported       feature_knn             1.0       0.9    0.9625    0.9775     0.968
                           graph_convolution       1.0       0.8      0.95    0.9325    0.9345
                           layer_propagation       1.0       0.5  0.797468  0.759067  0.778991
                           layer_vote              1.0       1.0    0.9625  0.974359     0.973
                           physical_partners       1.0       1.0       1.0  0.833333      0.98
                           random_forest           1.0       0.9    0.9625     0.975    0.9725
                           stacking                1.0       0.8    0.9375    0.9175    0.9205
                           stratum_focus           1.0       0.9    0.9625    0.9775     0.968
                           structural_homology     1.0  0.833333  0.882353  0.968421  0.959633
                           supervised_classifier   1.0       0.8      0.95      0.93    0.9255
                           triangulation           1.0       0.9    0.9625    0.9825   0.96997
                           understudied_first      1.0       0.9    0.9625    0.9825   0.96997
         lopit_pf_location cluster_guilt           1.0       0.0  0.277778  0.471831  0.405517
                           feature_knn             1.0  0.444444  0.671429  0.609091  0.599626
                           graph_convolution       1.0       0.6    0.6125    0.6325    0.5935
                           layer_propagation      0.75  0.111111  0.168831  0.158311  0.177402
                           layer_vote              1.0      0.45     0.625    0.5575      0.54
                           map_neighbours         0.75  0.315789  0.441176  0.431818  0.465023
                           physical_partners       NaN       NaN  0.333333  0.956522  0.852941
                           random_forest          0.75       0.5      0.75    0.6475    0.6525

## Whole classes hidden

Every gene of one class hidden at once. `right` is zero by construction: with every example of a label hidden, no strategy that learns from examples can name it. What is measured is `together` -- the share given one and the same label -- i.e. whether the strategy still recognises them as one thing, and `placed_at`, where it put them. Only sets with five or more answered members are ranked.

In [8]:
cls = led[(led['mode'] == 'set') & ~led['set_name'].astype(str).str.startswith('random')]
w = T.set_summary(cls)
assert (w['right'] == 0).all(), 'a hidden label was predicted: the class was not fully hidden'
w = w[w['answered'] >= 5]
w[['organism', 'strategy', 'set_name', 'set_size', 'answered', 'together', 'placed_at']].sort_values('together', ascending=False).head(20).round(2)

,organism,strategy,set_name,set_size,answered,together,placed_at
3,Tg,cluster_guilt,60S ribosome,40,12,1,40S ribosome
492,Tg,physical_partners,True,60,6,1,False
415,Tg,map_neighbours,True,99,99,1,False
414,Tg,map_neighbours,True,44,44,1,False
4,Tg,cluster_guilt,C,264,18,1,G1b
416,Tg,map_neighbours,True,60,60,1,False
394,Tg,map_neighbours,False,258,258,1,True
418,Tg,map_neighbours,True,44,44,1,False
417,Tg,map_neighbours,True,35,35,1,False
396,Tg,map_neighbours,False,274,274,1,True


## Your own list

`T.my_list(ctx, genes, target)` hides a pasted list together and asks every strategy. Two real labelled genes, as an example:

In [9]:
from starplast import strategies as S
ctx = S.Context.shipped('Tg')
rows = T.my_list(ctx, ['TGME49_294550', 'TGME49_244470'], 'compartment', strategies=['feature_knn', 'physical_partners', 'layer_vote'])
rows[['strategy', 'gene_id', 'truth', 'prediction', 'correct', 'abstained']]

,strategy,gene_id,truth,prediction,correct,abstained
0,feature_knn,TGME49_294550,nucleus - chromatin,nucleus - chromatin,True,False
1,feature_knn,TGME49_244470,apical 1,NaN,None,True
2,physical_partners,TGME49_294550,nucleus - chromatin,nucleus - chromatin,True,False
3,physical_partners,TGME49_244470,apical 1,apical 1,True,False
4,layer_vote,TGME49_294550,nucleus - chromatin,nucleus - chromatin,True,False
5,layer_vote,TGME49_244470,apical 1,apical 1,True,False
